# VQC Atlas · verified dataset to submission

Attach **VQC Verified Optimization Atlas** and the **Google Quantum AI – Verifiable Quantum Circuit Compilation** competition dataset. Use **Accelerator None**; Internet can be off. No previous notebook output is needed.

The atlas contains all 3,133,452 exact fusion targets, 33 search trials, 20 chemistry lookup attempts, labelled decoder examples and the complete V5 circuit portfolio. This notebook validates the package and creates `/kaggle/working/submission.zip`.

The selected circuits reproduce the V5 local projection of about **5.6634**, including one allowed decoder failure. This is not a new confirmed leaderboard score and the dataset itself does not guarantee an additional gain.

The default uses prepared circuits, rechecks the full integer oracle and fresh quantum/workspace samples, and reuses recorded complete checks only for exact file, native-library, verifier and scientific-reference matches. Changed dependencies trigger complete checks. Set the switches below to regenerate circuits or repeat all full checks explicitly.


In [ ]:
import os,sys,json,csv,zipfile
from pathlib import Path

FORCE_FULL_VERIFICATION = False
CHECK_FULL_INTEGER_ORACLE = True
REBUILD_ARITHMETIC_AND_LOOKUPS = False
REBUILD_DECODER = False  # enables the several-minute exact compiler pass too
WORKERS = min(4,os.cpu_count() or 1)

# Local overrides are optional; Kaggle discovers both inputs automatically.
INPUT = Path(os.environ.get('VQC_ATLAS_INPUT_ROOT','/kaggle/input'))
WORK = Path(os.environ.get('VQC_ATLAS_WORKDIR','/kaggle/working' if Path('/kaggle/working').exists() else str(Path.cwd()/'atlas_run'))).resolve()
WORK.mkdir(parents=True,exist_ok=True)
atlas_override=os.environ.get('VQC_ATLAS_ROOT')
comp_override=os.environ.get('VQC_COMPETITION')
if atlas_override:
    ATLAS=Path(atlas_override).resolve()
else:
    found=sorted(INPUT.rglob('atlas_manifest.json')) if INPUT.exists() else []
    if not found:
        # Fallback when an environment keeps the uploaded release as one ZIP.
        archives=list(INPUT.rglob('vqc_verified_optimization_atlas_v1.zip')) if INPUT.exists() else []
        if archives:
            extracted=WORK/'atlas_extracted';extracted.mkdir(exist_ok=True)
            with zipfile.ZipFile(archives[0]) as z:
                assert 'atlas_manifest.json' in z.namelist(),'Wrong dataset archive'
                for info in z.infolist():
                    p=Path(info.filename)
                    if p.is_absolute() or '..' in p.parts:raise ValueError('Unsafe archive member')
                z.extractall(extracted)
            found=[extracted/'atlas_manifest.json']
    if not found:raise FileNotFoundError('Attach VQC Verified Optimization Atlas. atlas_manifest.json was not found.')
    ATLAS=found[0].parent
if comp_override:
    COMP=Path(comp_override).resolve()
else:
    found=[p for p in INPUT.rglob('verify_submission_invsqrt.py')
           if p.parent.name=='invsqrt' and p.parent.parent.name=='verifiers'] if INPUT.exists() else []
    if not found:raise FileNotFoundError('Attach the official competition dataset as well as the atlas.')
    COMP=found[0].parent.parent.parent
sys.path[:0]=[str(ATLAS/'code'),str(COMP/'kickmix'),str(COMP/'verifiers')]
os.environ.update(OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1')
import numpy as np
import vqc_atlas as atlas
atlas.attach_competition(COMP)
manifest=atlas.load_manifest(ATLAS)
print('Atlas:',ATLAS,'\nCompetition:',COMP,'\nOutput:',WORK)
print('Dataset version:',manifest['schema_version'],'| CPU mode | Internet unnecessary')


**Inspect the measured portfolio and research corpus.** A successful arithmetic model, a resource estimate and a complete circuit check are different kinds of evidence. The dataset keeps them separate.

In [ ]:
from IPython.display import Markdown,display,FileLink
integrity=atlas.verify_integrity(ATLAS)
print('Integrity:',integrity)
profiles=list(csv.DictReader((ATLAS/'catalog/verified_profiles.csv').open()))
table='| Task | Qubits | Complete-check loss | Shots |\n|---|---:|---:|---:|\n'
for r in profiles:
    table+=f"| {r['task']} | {r['qubits']} | {float(r['cost']):,.3f} | {int(r['shots']):,} |\n"
display(Markdown(table))
trials=list(csv.DictReader((ATLAS/'experiments/fusion_search_trials.csv').open()))
layouts=list(csv.DictReader((ATLAS/'chemistry/lookup_layouts.csv').open()))
print('Fusion research trials:',len(trials),'| Chemistry design attempts:',len(layouts))
hard=json.loads((ATLAS/'dqi/hard_case_metadata.json').read_text())
print('Known decoder stress case, error weight:',hard['cases'][0]['error_weight'],
      '| reproduced on both decoder versions:',hard['batch_reproduced_on_both_circuits'])


**Use the data for new experiments.** Coefficients and exact targets are directly accessible. The example below inspects the smallest recorded phase margin. DQI examples include sparse error labels; chemistry tables provide the complete banked-QROM identity.

In [ ]:
with np.load(ATLAS/'fusion/selected_coefficients.npz',allow_pickle=False) as z:
    print('Selected coefficient shape:',z['table'].shape,'precision:',z['config'].tolist())
segments=list(csv.DictReader((ATLAS/'fusion/segment_certificates.csv').open()))
tight=min(segments,key=lambda r:int(r['minimum_margin_units']))
print('Tightest segment:',tight['table_index'],'x:',tight['tightest_margin_x'],
      'margin units:',tight['minimum_margin_units'])
with np.load(ATLAS/'chemistry/lookup_targets.npz',allow_pickle=False) as z:
    assert np.array_equal(z['even_words'] ^ z['xor_difference'], z['packed_deltas'][:,1::2])
    print('Complete chemistry delta table:',z['batch_deltas'].shape)
from atlas_data import check_dqi_labels
from dqi import verify_submission_dqi as D
import kickmix as km
with np.load(ATLAS/'dqi/weighted_examples.npz',allow_pickle=False) as z:
    cases={k:z[k] for k in z.files}
print('Sparse decoder labels:',check_dqi_labels(cases,D.FieldTables(km.GF2Field(12))))
del cases


In [ ]:
if REBUILD_ARITHMETIC_AND_LOOKUPS or REBUILD_DECODER:
    result=atlas.rebuild_selected(ATLAS,WORK/'rebuilt_selected',decoder=REBUILD_DECODER)
    print('Regenerated selected circuit bytes:',result)
else:
    print('Using prepared selected circuits. Set rebuild switches to reproduce them from source.')


**Validate and export.** Default checks include the complete integer target table, 32,768 fresh fusion circuit shots, 2,048 decoder shots, arbitrary-output ANGLE tests and the unchanged Hamiltonian accuracy calculation. Complete task records cover 200,540,928 fusion shots, 102,400 decoder shots and 254,464 chemistry shots. They are reused only for matching files and dependencies. Fresh private sampling may change the decoder penalty.

In [ ]:
validation=atlas.validate_selected(ATLAS,COMP,force_full=FORCE_FULL_VERIFICATION,
                                   workers=WORKERS,oracle=CHECK_FULL_INTEGER_ORACLE)
assert validation['status']=='PASS'
for task,report in validation['task_reports'].items():
    print(task,report['status'],'loss:',report['score'],'Q:',report['num_qubits'])
    print(report['verification_mode'])


In [ ]:
report=atlas.export_submission(ATLAS,WORK,validation)
print('New confirmed leaderboard score:',report['new_confirmed_lb'])
print('Local projection only:',round(report['local_projection'],6))
print('Submit:',str(WORK/'submission.zip'))
os.chdir(WORK)
display(FileLink('submission.zip'))
display(FileLink('atlas_validation_report.json'))


The exported archive has exactly 18 task files and is reopened, CRC-tested and checked against selected member hashes. Scientific reference data and verifier thresholds are unchanged. Full source attribution and component licensing are in the dataset's `licenses/`, `catalog/license_map.csv` and README. No private decoder generator is included or claimed.
